# 3 - QB - Panel à fréquences mixtes hétérogène

Ce notebook présente la création de jeux de données de séries temporelles et de panel à fréquences mixtes, en reprenant la logique du notebook `2 - QB - Mixed frequencies`.

Deux caractéristiques supplémentaires sont introduites pour le jeu de données de panel :
- Des **débuts et des fins de couverture différents pour chaque entité** (et non plus une période commune avec un simple historique tronqué pour une variable).
- Une **variable dont la fréquence de publication diffère selon l'entité** (annuelle pour certains pays, trimestrielle pour d'autres).

## 1 - Importation des modules

In [ ]:
# Importation des modules
# Modules de base
import warnings

# Manipulation de données
import numpy as np
import pandas as pd

# Graphiques
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

# Sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Module tsforecast
from tsforecast.frequency import HighFrequencyImputer
from tsforecast.frequency.target_frequency_validator import TargetFrequencyValidator
from tsforecast.frequency.frequency_aligner import FrequencyAligner
from tsforecast.frequency.imputation_window import ImputationWindowCalculator
from tsforecast.frequency.detector import detect_dataset_frequency, detect_index_frequency
from tsforecast.delays import PublicationDelayTransformer
from tsforecast.crossvals import TSOutOfSampleSplit

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Affichage
print("Modules importés avec succès !")

## 2 - Création des jeux de données

Nous allons créer deux jeux de données fictifs :
1. Un jeu de **séries temporelles** simples
2. Un jeu de **données de panel** (plusieurs entités)

Chaque jeu contient des variables à différentes fréquences (mensuelle, trimestrielle, annuelle) avec des délais de publication variables simulant des situations réelles.

### 2.1 - Jeu de données de séries temporelles

Ce jeu de données représente des indicateurs macroéconomiques typiques d'un pays :
- **PIB** : Publication trimestrielle avec délai de 2 mois
- **Inflation (IPC)** : Publication mensuelle avec délai de 1 mois
- **Taux de chômage** : Publication mensuelle avec délai de 1 mois
- **Production industrielle** : Publication mensuelle (disponible rapidement)
- **Balance commerciale annuelle** : Publication annuelle avec délai de 3 mois

In [ ]:
# Fonction de création de séries temporelles
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)

    # Création de l'index mensuel
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    # Initialisation du DataFrame
    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # ----- Variables mensuelles -----
    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC, entre 0.5% et 4%)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel, entre 5% et 12%)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),  # Choc économique
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # ----- Variable trimestrielle : PIB -----
    # Le PIB n'est disponible qu'aux fins de trimestre
    pib_base = 2500
    pib_growth_quarterly = 0.5  # Croissance trimestrielle moyenne
    df['pib_trimestriel'] = np.nan

    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # ----- Variable annuelle : Balance commerciale -----
    df['balance_commerciale_annuelle'] = np.nan

    for i, date in enumerate(dates):
        if date.month == 1:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # ----- Simulation des délais de publication -----
    # Délai de 1 mois pour l'inflation et le chômage
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    # Délai de 2 mois pour le PIB
    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    # Délai de 3 mois pour la balance commerciale annuelle
    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # ----- Simulation de données historiques limitées -----
    # La production industrielle n'est disponible qu'à partir de 2019
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

# Affichage
print("=" * 80)
print("JEU DE DONNÉES DE SÉRIES TEMPORELLES")
print("=" * 80)
print(f"\nPériode : {df_timeseries.index.min().strftime('%Y-%m')} à {df_timeseries.index.max().strftime('%Y-%m')}")
print(f"Nombre d'observations : {len(df_timeseries)}")
print(f"Colonnes : {list(df_timeseries.columns)}")
print("\n--- Statistiques descriptives ---")
print(df_timeseries.describe().round(2))
print("\n--- Aperçu des dernières lignes ---")
display(df_timeseries.tail(15))

### 2.2 - Jeu de données de panel

Ce jeu de données représente les mêmes indicateurs pour trois pays de la zone euro : France, Allemagne et Italie. Par rapport au notebook `2 - QB - Mixed frequencies`, deux différences sont introduites :

1. **Couverture temporelle hétérogène** : chaque pays a désormais son propre début et sa propre fin de série (et non plus une période commune de 2018-01 à 2024-07 avec un simple historique tronqué pour la production industrielle).
2. **Fréquence de publication hétérogène pour une même variable** : les dépenses publiques (`depenses_publiques_pib`) sont publiées annuellement pour la France et l'Italie, mais trimestriellement pour l'Allemagne.

In [ ]:
# Fonction de création d'un jeu de données de panel fictif
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)

    # Définition des pays et leurs caractéristiques
    # Chaque pays possède sa propre période de couverture (start_date / end_date)
    # ainsi que sa propre fréquence de publication pour les dépenses publiques
    countries = {
        'France': {
            'pib_base': 2800,
            'inflation_base': 1.5,
            'chomage_base': 8.0,
            'depenses_base': 55.0,
            'start_date': '2018-01-01',
            'end_date': '2024-07-01',
            'prod_ind_start': '2018-06-01',  # Historique complet
            'depenses_frequency': 'annuelle'
        },
        'Allemagne': {
            'pib_base': 3500,
            'inflation_base': 1.2,
            'chomage_base': 5.5,
            'depenses_base': 45.0,
            'start_date': '2018-07-01',  # Début plus tardif que la France
            'end_date': '2024-04-01',  # Fin plus précoce que la France
            'prod_ind_start': '2019-01-01',  # Historique partiel
            'depenses_frequency': 'trimestrielle'
        },
        'Italie': {
            'pib_base': 2200,
            'inflation_base': 1.8,
            'chomage_base': 10.5,
            'depenses_base': 50.0,
            'start_date': '2019-01-01',  # Début encore plus tardif
            'end_date': '2024-07-01',
            'prod_ind_start': '2019-06-01',  # Historique plus court
            'depenses_frequency': 'annuelle'
        }
    }

    # Initialisation de la liste des jeux de données pour l'ensemble des pays
    all_data = []

    # Parcours des pays
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        # Création de l'index de dates propre à ce pays (début/fin distincts)
        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        # Création du DataFrame pour ce pays
        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        # Production industrielle (mensuelle)
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise

        # Données non disponibles avant une certaine date
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        # Inflation (mensuelle)
        infl_trend = np.linspace(
            params['inflation_base'],
            params['inflation_base'] + np.random.uniform(0.5, 2.0),
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        # Taux de chômage (mensuel)
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Balance commerciale annuelle
        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 1:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Dépenses publiques (% du PIB) : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Simulation des délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    # Concaténation et création du MultiIndex
    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

# Affichage
print("=" * 80)
print("JEU DE DONNÉES DE PANEL")
print("=" * 80)
print(f"\nEntités (pays) : {df_panel.index.get_level_values('country').unique().tolist()}")
print(f"Colonnes : {list(df_panel.columns)}")
print(f"Shape : {df_panel.shape}")

print("\n--- Période couverte par entité ---")
for country in df_panel.index.get_level_values('country').unique():
    dates_country = df_panel.loc[country].index
    print(f"  {country} : {dates_country.min().strftime('%Y-%m')} à {dates_country.max().strftime('%Y-%m')}")

print("\n--- Statistiques par pays ---")
display(df_panel.groupby('country').agg(['count', 'mean']).round(2))
print("\n--- Aperçu pour la France ---")
display(df_panel.loc['France'].tail(10))

### 2.3 - Vérification des deux caractéristiques ajoutées

On vérifie ci-dessous que chaque entité a bien sa propre période de couverture, et que la variable `depenses_publiques_pib` est bien publiée à des fréquences différentes selon les pays.

In [ ]:
# Vérification de la fréquence de publication des dépenses publiques par entité
print("--- Fréquence de publication de 'depenses_publiques_pib' par entité ---")
for country in df_panel.index.get_level_values('country').unique():
    serie = df_panel.loc[country, 'depenses_publiques_pib'].dropna()
    if len(serie) > 1:
        avg_gap_months = round((serie.index[1:] - serie.index[:-1]).mean().days / 30)
        freq_label = 'Annuelle' if avg_gap_months >= 9 else 'Trimestrielle'
    else:
        freq_label = 'Indéterminée'
    print(f"  {country} : {freq_label} ({len(serie)} observations, écart moyen ~{avg_gap_months} mois)")